[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day9_live.ipynb)

# Day 9 · 강의 — 에이전트의 구조

설비 일지를 대신 찾아 주는 비서를 만든다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

모든 셀에 **코드가 채워져 있다.** 위에서부터 실행해 결과를 눈으로 확인한다.
강사가 설명하는 동안 값을 바꿔 가며 돌려 본다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 준비

오늘 만드는 것은 **설비 일지를 대신 찾아 주는 비서**다.

「3호기 어제 불량률이 2 퍼센트 넘었어?」 처럼 물으면, 모델이 스스로 사내 기록을 조회하고
필요하면 계산까지 해서 답한다. 끝까지 가면 도구를 하나 더 붙이는 데 **세 줄**이면 된다.

어제 쓰던 `build.nvidia.com` 키를 그대로 쓴다.

1. `build.nvidia.com` 에 접속해 로그인한다
2. 아무 모델이나 열고 **Get API Key** 를 누른다
3. `nvapi-` 로 시작하는 키를 복사해 아래 셀을 실행한 뒤 입력창에 붙여 넣는다

In [ ]:
# 키는 화면에 안 찍히게 받는다. 붙여 넣고 Enter 를 누르면 된다.
import getpass, json, urllib.request
KEY = getpass.getpass('nvapi- 로 시작하는 키: ')
print('키 길이', len(KEY))     # 60~80 정도면 제대로 들어간 것이다

In [ ]:
# 모델에 대화를 통째로 보내는 함수. 실패해도 노트북이 멈추지 않게 [실패] 를 돌려준다.
URL = 'https://integrate.api.nvidia.com/v1/chat/completions'
MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1'

def chat(messages, tools=None, n=400, temp=0):
    body = {'model': MODEL, 'max_tokens': n, 'temperature': temp,
            'messages': messages}
    if tools:                       # 도구 목록은 있을 때만 같이 보낸다
        body['tools'] = tools
    req = urllib.request.Request(URL, data=json.dumps(body).encode(), headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    for _ in range(2):                  # 붐빌 때가 있어 한 번 더 시도한다
        try:
            with urllib.request.urlopen(req, timeout=180) as f:
                return json.load(f)['choices'][0]['message']
        except Exception as e:
            err = str(e)[:80]
    return {'role': 'assistant', 'content': '[실패] %s' % err}

In [ ]:
# 한 문장만 물어볼 때 쓰는 짧은 이름
def say(text, n=200):
    return (chat([{'role': 'user', 'content': text}], n=n).get('content') or '').strip()

print(say('한 단어로만 답하라. 대한민국의 수도는?', 10))   # '서울' 이 나오면 준비 끝이다

> `[실패]` 가 나오면 키를 잘못 붙였거나 모델이 붐비는 것이다. 셀을 다시 실행해 본다.

## 2. 도구 없이 물어보면

먼저 **모델 혼자서는 어디까지 되는지** 본다. 성격이 다른 세 가지를 물어본다.

In [ ]:
# 세 가지를 그냥 물어본다
for q in ['17 곱하기 24 는 얼마인가? 숫자만 답하라.',
          '오늘 날짜는? 날짜만 답하라.',
          '3호기의 어제 불량률은 몇 퍼센트인가?']:
    print('Q', q)
    print('A', say(q, 60).replace('\n', ' ')[:100])
    print()

세 답이 전부 다르게 나온다.

| 질문 | 이 모델은 | 왜 |
|---|---|---|
| 17 × 24 | **맞힌다** | 이 크기면 두 자리는 된다. 자릿수를 키우면 갈린다 |
| 오늘 날짜 | **옛날 날짜**를 자신 있게 답한다 | 학습이 끝난 시점에 멈춰 있다 |
| 3호기 불량률 | **되묻는다** | 사내 기록이라 배운 적이 없다 |

가운데가 제일 위험하다. **틀린 답이 맞는 답과 똑같은 말투로 나온다.**
셋 다 원인은 하나다 &mdash; 모델 안에 그 답이 없다.

## 3. 계산 하나는 프롬프트로도 나아진다

도구로 넘어가기 전에, **프롬프트만으로 어디까지 되는지** 먼저 본다.
한 줄씩 쓰게 하면 어려운 곱셈이 쉬운 덧셈으로 갈린다.
두 자리 곱셈은 이 크기 모델이면 그냥도 맞힌다. 갈리는 자리는 그 다음이다.

In [ ]:
# 그냥 물었을 때와 자리별로 나누게 했을 때
print('[그냥]      ', say('17 곱하기 24 는? 숫자만 답하라.', 30).replace('\n', ' '))
print('[자리별로]  ', say('17 곱하기 24 를 자리별로 나눠 한 줄씩 계산하고 마지막 줄에 답만 써라.', 200)
      .replace('\n', ' / ')[:160])

**단계를 다 펼쳐 놓고도 마지막 합에서 틀린다.** 중간 줄이 그럴듯해서 더 안 보인다.
맞았는지 확인할 방법이 없다는 것이 더 곤란하다. 정확한 값은 **계산기에 넘긴다**.

> **실습문제 1.** 곱하는 두 수를 **네 자리 × 세 자리**로 키워 다시 돌려 본다.
> `___` 자리에 `4271 곱하기 386` 처럼 넣으면 된다. 아래 `check` 가 정답을 같이 찍어 준다.

In [ ]:
# 자릿수를 키우면 같은 방법이 계속 통하는지 보는 것이다
q = '4271 곱하기 386 을 자리별로 나눠 한 줄씩 계산하고 마지막 줄에 답만 써라.'

print(say(q, 300))
print('정답', 4271 * 386)

## 4. 도구를 만들어 준다

도구는 특별한 것이 아니다. **평범한 파이썬 함수**다.
지금은 사내 기록 대신 표 하나를 코드 안에 넣어 두고 쓴다. 현업에서는 이 자리에 사내 DB 조회나 엑셀 읽기가 들어간다.

In [ ]:
# 어제 하루치 설비 일지 — 현업에서는 이 자리가 DB 조회나 엑셀 읽기가 된다
LOG = {
    '1호기': {'라인': 'A', '생산': 1240, '불량': 30, '근무조': '주간'},
    '2호기': {'라인': 'A', '생산':  980, '불량': 11, '근무조': '야간'},
    '3호기': {'라인': 'B', '생산': 1530, '불량': 28, '근무조': '주간'},
    '4호기': {'라인': 'B', '생산':  760, '불량': 24, '근무조': '야간'},
}
for k, v in LOG.items():
    print('%s  %s라인  생산 %5d  불량 %3d' % (k, v['라인'], v['생산'], v['불량']))

In [ ]:
# 함수 셋 — 계산기 · 오늘 날짜 · 설비 조회
import datetime

def calc(expr):
    """산술식 하나를 계산해 문자열로 돌려준다"""
    try:
        return str(eval(expr, {'__builtins__': {}}, {}))
    except Exception:                       # 죽지 말고 고칠 방법을 알려 준다
        return '계산할 수 없다. 숫자와 + - * / ( ) 만 넣어라. 예: 28/1530*100'

def today():
    """오늘 날짜"""
    return datetime.date.today().isoformat()

def machine_info(machine):
    """설비 한 대의 어제 기록. 없는 이름이면 쓸 수 있는 이름을 알려 준다."""
    v = LOG.get(machine)
    if v is None:
        return '그런 설비는 없다. 쓸 수 있는 이름: ' + ', '.join(LOG)
    return '%s: %s라인, 생산 %d개, 불량 %d개, 근무조 %s' % (
        machine, v['라인'], v['생산'], v['불량'], v['근무조'])

FUNCS = {'calc': calc, 'today': today, 'machine_info': machine_info}
print(calc('17*24'))
print(today())
print(machine_info('3호기'))
print(machine_info('9호기'))     # 없는 이름을 넣으면 이렇게 알려 준다

모델은 이 함수를 **볼 수 없다**. 이름 · 설명 · 인자 모양만 글로 건네받는다.
그 설명이 곧 모델용 프롬프트다. 애매하게 적으면 도구를 안 부르거나 엉뚱하게 부른다.

In [ ]:
# 모델에게 건네는 도구 목록. spec() 은 매번 같은 모양을 찍어 주는 짧은 도우미다.
def spec(name, desc, props, required):
    return {'type': 'function', 'function': {
        'name': name, 'description': desc,
        'parameters': {'type': 'object', 'properties': props, 'required': required}}}

TOOLS = [
    spec('calc', '산술식을 계산한다. 나눗셈·퍼센트처럼 정확한 값이 필요할 때 쓴다.',
         {'expr': {'type': 'string', 'description': '파이썬 산술식. 예: 28/1530*100'}}, ['expr']),
    spec('today', '오늘 날짜를 YYYY-MM-DD 로 돌려준다.', {}, []),
    spec('machine_info', '설비 한 대의 어제 생산량·불량 수·라인·근무조를 사내 일지에서 찾아 돌려준다.',
         {'machine': {'type': 'string', 'description': '설비 이름. 예: 3호기'}}, ['machine']),
]
print(json.dumps(TOOLS[2], ensure_ascii=False, indent=1))

## 5. 모델이 도구를 고른다

도구 목록을 같이 보내면 무엇이 달라지는지 본다.

In [ ]:
# 도구 목록을 같이 보내면 답 대신 '무엇을 부를지' 가 돌아온다
m = chat([{'role': 'user', 'content': '3호기 어제 불량률은?'}], TOOLS, 200)
print('내용     ', m.get('content'))
print('도구 호출', m.get('tool_calls'))

`content` 가 비고 `tool_calls` 가 찬다. **모델이 고른 것은 답이 아니라 도구**다.
실제로 부르는 것은 우리 쪽 코드다. 결과를 다시 넣어 줘야 비로소 답이 나온다.

> `tool_calls` 가 계속 `None` 이면 그 모델이 도구 호출을 안 받는 것이다.
> 위 준비 셀의 `MODEL` 을 `meta/llama-3.3-70b-instruct` 로 바꿔 다시 실행한다.

## 6. 루프 — 판단 · 행동 · 관찰

도구를 부르고, 결과를 되먹이고, 다시 묻는다. **더 부를 것이 없을 때까지** 도는 것이 에이전트다.
아래가 그 전부다. 열 줄 남짓이고, 오늘 뒤에 나오는 것들은 전부 이 함수 주변에 붙는다.

In [ ]:
# 에이전트 본체. 마지막 대화는 LAST 에 남겨 두었다가 9절에서 다시 본다.
SYSTEM = ('너는 공정 데이터 비서다. 필요하면 도구를 부르고, 모르면 모른다고 답한다. '
          '한국어로만 답한다.')   # 이 한 줄이 없으면 중국어 낱말이 섞여 나오기도 한다
LAST = []

def run_agent(question, system=SYSTEM, max_steps=5, log=True):
    global LAST
    messages = [{'role': 'system', 'content': system},
                {'role': 'user', 'content': question}]
    for step in range(max_steps):
        m = chat(messages, TOOLS, 500)           # ① 판단 — 부를까, 답할까
        messages.append(m)
        calls = m.get('tool_calls') or []
        if not calls:                            # 부를 것이 없으면 그것이 답이다
            LAST = messages
            return m.get('content') or ''
        for c in calls:                          # ② 행동 — 고른 도구를 실행
            name = c['function']['name']
            args = json.loads(c['function']['arguments'] or '{}')
            try:
                out = FUNCS[name](**args)
            except Exception as e:           # 에러도 결과처럼 되먹인다
                out = '오류: %s' % e
            if log:
                print('  [도구] %s(%s) -> %s' % (name, args, out))
            messages.append({'role': 'tool', 'tool_call_id': c['id'],
                             'content': out})    # ③ 관찰 — 결과를 대화에 되먹인다
    LAST = messages
    return '[한도] %d번 안에 못 끝냈다' % max_steps

In [ ]:
# 도구가 필요한 질문
print(run_agent('3호기 어제 불량률은 몇 퍼센트야?'))

`[도구]` 줄이 실제로 부른 기록이다. 조회로 숫자를 가져오고, 계산기로 퍼센트를 냈다면 두 줄이 찍힌다.

In [ ]:
# 도구가 필요 없는 질문 — [도구] 줄이 안 찍힌다
print(run_agent('안녕? 너는 무슨 일을 하니?'))

좋은 에이전트는 **도구를 안 쓸 때도 안다**. 상식 질문까지 도구를 부르면 느리고 비싸진다.

## 7. 멀티스텝 — 앞 결과가 있어야 다음을 부른다

한 번에 안 끝나는 질문을 준다. 앞 도구의 결과를 봐야 다음 도구를 정할 수 있는 것들이다.

In [ ]:
# 조회 → 비교
print(run_agent('3호기 어제 불량률이 2 퍼센트보다 높았어?'))

`[도구]` 가 한 줄만 찍히기도 한다. 나눗셈이 쉬우면 **모델이 그냥 계산해 버린다**.
그래도 **조회는 반드시 먼저** 한다 &mdash; 생산량과 불량 수는 지어낼 수가 없기 때문이다.

In [ ]:
# 조회 네 번 → 집계
print(run_agent('1호기부터 4호기까지 어제 불량률을 모두 구해서 가장 높은 설비를 알려줘'))

`[도구]` 줄이 **두 번 이상** 찍히면 멀티스텝이다. 앞 결과를 보고 다음 도구를 정했다는 뜻이다.
이 부분이 「미리 순서를 정해 둔 코드」와 갈리는 지점이다.

도구가 **쓸 수 있는 이름을 알려 주는 에러**를 돌려주면, 모델은 그걸 읽고 다시 고른다.
`KeyError` 로 죽는 도구였다면 여기서 루프가 끝났을 것이다. **에러 문구가 곧 다음 행동의 힌트**다.

> **실습문제 2.** **없는 설비 이름**을 넣어 물어본다. 에이전트가 어떻게 빠져나오는지 본다.
> 쓸 수 있는 이름은 1호기 · 2호기 · 3호기 · 4호기 뿐이다. `7호기` 처럼 없는 것을 넣어 본다.

In [ ]:
# 도구가 던지는 에러 문구가 다음 행동을 어떻게 바꾸는지 보는 것이다
ans = run_agent('7호기의 어제 불량률을 알려줘')

print(ans)

## 8. 안전장치

루프는 스스로 멈추지 않을 수 있다. 그래서 **한도**를 같이 만든다.
`max_steps` 를 줄이면 도중에 끊긴다는 것을 먼저 확인한다.

In [ ]:
# 한도를 1로 줄이면 도구를 한 번 부르고 끝난다
print(run_agent('1호기부터 4호기까지 평균 불량률을 알려줘', max_steps=1))

현업에서는 여기에 두 가지를 더 붙인다.
**되돌릴 수 없는 도구**(삭제 · 발주 · 메일 발송)는 실행 전에 사람에게 묻고,
**부른 도구와 인자를 전부 로그로 남긴다**. 안 보이면 못 고친다.

## 9. 컨텍스트 — 대화가 얼마나 커지는가

모델은 지난 대화를 가지고 있지 않다. **매 호출마다 목록 전체를 다시 보낸다.**
그래서 도구가 돌려준 것이 쌓이면 보내는 양이 계속 커진다. 방금 대화로 직접 세어 본다.

In [ ]:
# 방금 대화에 무엇이 들어 있는지 본다
for m in LAST:
    print('%-9s %s' % (m['role'], str(m.get('content'))[:70]))
print()
print('메시지 %d개 · 글자 %d자' % (len(LAST), len(json.dumps(LAST, ensure_ascii=False))))

In [ ]:
# 도구를 많이 부르는 질문일수록 커진다
run_agent('1호기부터 4호기까지 불량률을 모두 구해서 라인별로 정리해줘', log=False)
print('메시지 %d개 · 글자 %d자' % (len(LAST), len(json.dumps(LAST, ensure_ascii=False))))

늘어나는 것은 사람이 친 말이 아니라 **도구가 돌려준 것**이다. 줄일 자리도 거기다.

### 줄이는 법 — 요약해서 넘긴다

In [ ]:
# 대화를 글로 펼쳐 요약을 받는다
def flatten(messages):
    return '\n'.join('%s: %s' % (m['role'], str(m.get('content'))[:200])
                     for m in messages)

In [ ]:
# 정한 것과 못 푼 것만 남긴다
brief = say('아래 대화를 세 줄로 요약하라. 정한 것과 아직 못 푼 것만 남기고 중복은 버려라.\n\n'
            + flatten(LAST), 300)
print(brief)
print()
print('원본 %d자 -> 요약 %d자' % (len(json.dumps(LAST, ensure_ascii=False)), len(brief)))

> 이 셀은 보내는 글이 길어서 가끔 `[실패] timed out` 이 뜬다. 그때는 다시 실행하면 된다.

요약은 **되돌릴 수 없다**. 무엇을 남길지 미리 정해 두지 않으면 필요한 것부터 사라진다.
그래서 실무에서는 「정한 것 · 못 푼 것 · 파일 경로」처럼 **남길 항목을 먼저 정해 두고** 요약시킨다.

## 10. 내 업무로

여기서부터는 각자 자기 업무로 바꾼다. **루프 코드는 손대지 않는다.**
바꾸는 것은 시스템 프롬프트 한 줄, 데이터, 도구 설명뿐이다.

**도구를 늘려도 루프는 그대로다.** 바뀌는 것은 목록과 설명뿐이다.
그래서 에이전트를 키우는 일은 코드를 늘리는 일이 아니라 **도구를 정리하는 일**이 된다.

## 11. 반출본 만들기 — 컬럼 이름부터

여기서부터는 **공정 데이터를 Codex 에 어떻게 넘기느냐**의 문제다.

개인정보라면 이름·사번 같은 식별자만 떼면 됐다. 공정 데이터는 반대다.
코팅 로딩, 전극 밀도, 화성 전압 — **그 숫자 자체가 레시피**라 뗄 식별자가 없다.
그래서 값을 지우는 대신 **절대값을 안 넘기는 형태로 바꿔서** 넘긴다.

In [ ]:
# 셀 공정 기록 2,400행을 읽어 온다. 난수로 만든 가상 데이터라 이 파일 자체는 반출 걱정이 없다.
import pandas as pd, numpy as np

URL = 'https://tunalee.github.io/posco/data/cell_process.csv'
df = pd.read_csv(URL, parse_dates=['시각'])
print(df.shape)
print(df.head(3).to_string(index=False))

### 이대로 붙이면 무엇이 나가나

In [ ]:
# 컬럼 이름만으로도, 값 몇 줄만으로도, 로트 번호와 시각만으로도 새어 나간다
print(list(df.columns))
print()
print(df[['건조_ZONE2_TEMP', '코팅_로딩_mg_cm2', '전극_밀도']].head(3).to_string(index=False))
print()
print('로트', df['로트번호'].iloc[0], '~', df['로트번호'].iloc[-1], '· 총', df['로트번호'].nunique(), '개')
print('가장 흔한 간격', df['시각'].diff().mode()[0])

`화성_3단계_CV_전압` 하나로 **화성 공정이 몇 단인지, 어느 단이 CV 구간인지**가 드러난다.
`NMP_투입비` 는 슬러리 배합이고, `코팅_로딩_mg_cm2` 와 `전극_밀도` 는 셀 설계값이다.
**값보다 컬럼명이 먼저 샌다.**

### 내보내기 전에 표부터 손본다

이 표에는 실제 라인에서 흔한 사고가 일부러 들어 있다. **반출본을 만들기 전에** 잡아 둔다.
안 잡고 변환하면 평균과 표준편차가 오염돼서 반출본 전체가 틀어진다.

In [ ]:
# 어디에 무엇이 있는지 훑는다
print('결측       ', df.isna().sum()[lambda s: s > 0].to_dict())
run = df['프레스_1호기_압력'].groupby((df['프레스_1호기_압력'].diff() != 0).cumsum()).size()
print('같은 값 연속', run.max(), '행')
print('ZONE1 범위 ', df['건조_ZONE1_TEMP'].min(), '~', df['건조_ZONE1_TEMP'].max())
print('시각 역행  ', (df['시각'].diff().dt.total_seconds() < 0).sum(), '건')
print('시각 중복  ', df['시각'].duplicated().sum(), '건')

네 가지가 그대로 보인다.

| 무엇 | 어디에 | 왜 생기나 |
|---|---|---|
| 연속 결측 40행 | `건조_ZONE2_TEMP` | 센서 단선 |
| 같은 값 60행 | `프레스_1호기_압력` | 값 고착 — 센서가 멎었다 |
| 최대 278.6 | `건조_ZONE1_TEMP` | 화씨가 섞여 들어왔다 |
| 역행 3건 · 중복 15건 | `시각` | 수집기 재시작 |

In [ ]:
# 세 가지를 한 번에 손본다. 이 표를 고친 다음에야 반출본을 만든다.
mask = df['건조_ZONE1_TEMP'] > 150                      # 화씨로 들어온 구간
df.loc[mask, '건조_ZONE1_TEMP'] = ((df.loc[mask, '건조_ZONE1_TEMP'] - 32) * 5 / 9).round(1)

same = df['프레스_1호기_압력'].diff() == 0               # 값 고착 — 그대로 두면 평균이 끌려간다
df.loc[same, '프레스_1호기_압력'] = np.nan

df = df.sort_values('시각').drop_duplicates('시각').reset_index(drop=True)   # 시각 사고
print('화씨 %d행 되돌림 · 고착 %d행 결측 처리 · 정렬·중복 제거 후 %d행'
      % (mask.sum(), same.sum(), len(df)))

### 이름 규칙 — 물리량은 남기고 공정 맥락은 지운다

`FEAT_017` 처럼 다 지우면 모델이 **무슨 값인지 몰라** 코드 품질이 떨어진다.
그래서 앞부분에 **물리량 종류**만 남기고, 뒷부분의 공정 이름을 순번으로 바꾼다.

| 원본 | 익명명 | 살아 있는 정보 |
|---|---|---|
| 건조_ZONE2_TEMP | `TEMP_D2` | 온도끼리 같이 볼 만하다 |
| 프레스_1호기_압력 | `PRES_B1` | 압력이다 |
| 화성_3단계_CV_전압 | `VOLT_P3` | 전압이다 |
| NMP_투입비 | `RATIO_M2` | 비율이라 합이 1일 수 있다 |

In [ ]:
# 컬럼마다 (익명명, 변환 방식, 파라미터) 를 정해 둔 표. 이 표가 곧 반출 스크립트 명세다.
#   spec : (x - target) / tol   — 스펙이 있는 값. 관리도·Cpk 가 그대로 나온다
#   z    : (x - 평균) / 표준편차 — 스펙을 모를 때. 상관·회귀가 그대로 나온다
#   rank : 백분위                — 순서만 남긴다. 회귀에는 못 쓴다
RULE = {
    '건조_ZONE2_TEMP':   ('TEMP_D2',  'spec', (120.0, 5.0)),
    '프레스_1호기_압력':  ('PRES_B1',  'z',    None),
    '코팅_로딩_mg_cm2':  ('LOAD_C1',  'z',    None),
    '화성_3단계_CV_전압': ('VOLT_P3',  'z',    None),
    'NMP_투입비':        ('RATIO_M2', 'rank', None),
}
for src, (dst, how, p) in RULE.items():
    print('%-18s -> %-9s %s' % (src, dst, how))

### 숫자가 아닌 열은 다르게 다룬다

여기까지가 **연속형** 열이다. 표에는 다른 성격의 열도 있다.

| 성격 | 이 표에서는 | 처리 | 왜 |
|---|---|---|---|
| 범주형 · 몇 개뿐 | 설비호기 · 교대조 | **원핫** | 1·2·3·4 로 두면 4호기가 2호기의 두 배가 된다 |
| 식별자 | 로트번호 | **재번호** | 순번에서 생산량이 역산된다 |
| 시각 | 시각 | **t0 기준 상대 시간** | 간격에서 tact time 이 샌다 |
| 목표 | 판정 | **0 / 1** | 맞혀야 할 칸이라 바꾸지 않는다 |

In [ ]:
# 표대로 바꿔 주는 함수. 여기 코드는 안 고친다 — 위의 RULE 과 아래 목록만 고친다.
ONEHOT = ['설비호기', '교대조']          # 종류가 적은 범주형
IDCOL, TIMECOL, TARGET = '로트번호', '시각', '판정'

def export(df, rule):
    out = pd.DataFrame(index=df.index)
    for src, (dst, how, p) in rule.items():          # ① 연속형
        x = df[src].astype(float)
        if   how == 'spec': out[dst] = ((x - p[0]) / p[1]).round(3)
        elif how == 'z':    out[dst] = ((x - x.mean()) / x.std()).round(3)
        elif how == 'rank': out[dst] = x.rank(pct=True).round(3)
    for i, c in enumerate(ONEHOT):                   # ② 범주형 — 열을 가른다
        d = pd.get_dummies(df[c], prefix='CAT%d' % (i + 1)).astype(int)
        d.columns = ['CAT%d_%d' % (i + 1, k) for k in range(d.shape[1])]
        out = pd.concat([out, d], axis=1)
    out['LOT'] = pd.factorize(df[IDCOL])[0]          # ③ 식별자 — 순번을 지운다
    out['T']   = ((df[TIMECOL] - df[TIMECOL].min())
                  .dt.total_seconds() / 3600).round(2)   # ④ 시각 — t0 기준
    out['OK']  = (df[TARGET] == '양품').astype(int)  # ⑤ 목표 — 그대로 둔다
    return out

In [ ]:
# 반출본을 만들고, 검사를 통과하면 파일로 남긴다
out = export(df, RULE)
print(out.head(3).to_string(index=False))

이제 `-0.32` 만 보고는 건조 온도가 **80도대인지 120도대인지 알 수 없다**.
`LOT` 은 0 부터 다시 매겨 생산량이 안 보이고, `T` 는 첫 행을 0 으로 잡은 상대 시간이다.

### 나가도 되는 모양인지 자동으로 본다

In [ ]:
# 사람 눈으로 매번 보지 않는다. 검사기를 하나 만들어 두고 그것만 통과시킨다.
def check(out, src, rule):
    bad = []
    ko = [c for c in out.columns if any('가' <= ch <= '힣' for ch in c)]
    if ko:
        bad.append('한글 컬럼명이 남았다: %s' % ko)
    same = [c for c in out.columns if c in src.columns]
    if same:
        bad.append('원본 컬럼명이 그대로다: %s' % same)
    for c in out.columns:
        for s in src.select_dtypes('number').columns:
            if out[c].round(3).equals(src[s].round(3)):
                bad.append('%s 가 원본 %s 와 같은 값이다' % (c, s))
    print('\n'.join(bad) if bad else '내보내도 되는 모양이다')

check(out, df, RULE)

### 통과했으면 파일로 남긴다

반출본은 **한 번 만들어 파일로 굳힌다**. 매번 다시 만들면 그때그때 달라진다.
z-score 는 평균과 표준편차로 계산하니까, 행이 하나만 늘어도 값이 전부 조금씩 바뀐다.

In [ ]:
# 반출본 CSV 와 매핑 표를 나눠 저장한다
out.to_csv('export.csv', index=False)                 # 이것만 반출 대상이다
pd.DataFrame([(s_, d, h, str(p)) for s_, (d, h, p) in RULE.items()],
             columns=['원본명', '익명명', '방식', '파라미터']).to_csv('mapping_local.csv', index=False)
print('export.csv %d행 %d열  ·  mapping_local.csv %d행' % (*out.shape, len(RULE)))
print('mapping_local.csv 는 사내에만 둔다')

파일이 둘로 갈린다. **`export.csv` 는 나가도 되고, `mapping_local.csv` 는 절대 안 된다.**
이름을 그렇게 지어 두면 실수로 같이 올리는 일이 준다.

### 분석 결과가 정말 그대로 나오는지

In [ ]:
# 원본에서 잰 상관과 반출본에서 잰 상관을 견준다
pair = [('건조_ZONE2_TEMP', 'TEMP_D2'), ('프레스_1호기_압력', 'PRES_B1'),
        ('코팅_로딩_mg_cm2', 'LOAD_C1')]
a = df[[s for s, _ in pair]].corr().round(3).values
b = out[[d for _, d in pair]].corr().round(3).values
print('원본 상관\n', a)
print('반출본 상관\n', b)
print('같은가:', (a == b).all())

**같다.** 상관·회귀·관리도 판정·이상탐지·PCA 는 값을 선형으로 옮겨도 결과가 안 바뀐다.
그래서 절대값을 안 넘기고도 분석 코드는 그대로 만들 수 있다.

이 표가 **반출 스크립트 명세**다. 한 번 정해 두면 컬럼이 늘어도 매번 판단할 일이 없다.

### 여기까지가 사람이 하는 일

가명처리 · 정규화 · 인코딩 · 재번호는 **손으로 먼저 끝낸다**. Codex 는 그 다음부터다.
넘길 것은 값이 아니라 **반출본의 스키마**다. 아래 셀이 그 스키마를 프롬프트로 만들어 준다.

In [ ]:
# 반출본을 보고 Codex 에 붙일 프롬프트를 만든다
def make_prompt(out, goal, done, todo):
    lines = []
    for c in out.columns:
        v = out[c]
        lines.append('%-10s %-7s 결측%-4d 범위 %.2f~%.2f'
                     % (c, v.dtype, v.isna().sum(), v.min(), v.max()))
    return ('# 역할\n너는 공정 데이터로 %s 코드를 쓰는 사람이다.\n\n'
            '# 데이터 — 값은 못 준다. 반출본의 모양만 준다\n행 %d · 열 %d\n%s\n\n'
            '# 이미 끝난 것 — 다시 하지 마라\n%s\n\n'
            '# 남은 것 — 이것만 해라\n%s\n\n'
            '# 형식\n바로 돌아가는 파이썬 코드로 준다. 데이터를 출력하는 줄은 넣지 마라.'
            % (goal, len(out), out.shape[1], '\n'.join(lines), done, todo))

In [ ]:
# 그대로 복사해 Codex 에 붙이면 된다
PROMPT = make_prompt(
    out, goal='OK 열(1 양품 · 0 불량)을 맞히는',
    done='컬럼 가명처리 · 스펙 정규화와 z-score · 원핫 인코딩 · 로트 재번호 · 상대시간',
    todo='결측 처리 · 학습과 검증 분할 · 모델 둘 비교 · 놓친 불량 세기')
print(PROMPT)

**「이미 끝난 것」을 안 적으면** 정규화와 인코딩을 한 번 더 하는 코드가 온다.
**「데이터를 출력하는 줄은 넣지 마라」를 안 적으면** `df.head()` 를 찍는 코드가 온다.
둘 다 한 줄이면 막힌다.

> **실습문제 3.** `RULE` 에서 `NMP_투입비` 의 방식을 `'rank'` 에서 `'z'` 로 바꾸고 다시 만들어 본다.
> 방식 이름 한 글자만 바꾸면 된다. 함수는 안 고친다.
> 분위 변환은 순서만 남기고 선형성이 깨져서 회귀에 못 쓴다는 것을 눈으로 본다.

In [ ]:
# RULE 의 값 부분만 바꾸는 문제다
RULE['NMP_투입비'] = ('RATIO_M2', 'z', None)

out2 = export(df, RULE)
print('rank 일 때 상관', round(out['RATIO_M2'].corr(out['TEMP_D2']), 3))
print('z 일 때 상관   ', round(out2['RATIO_M2'].corr(out2['TEMP_D2']), 3))
print('원본 상관      ', round(df['NMP_투입비'].corr(df['건조_ZONE2_TEMP']), 3))

## 12. 반출본으로 모델까지

여기서부터는 **Codex 가 준 코드를 사내에서 돌리는 자리**다.
앞 절에서 만든 `out` 하나로 학습부터 판정까지 간다. 원본 `df` 는 더 안 쓴다.

In [ ]:
# 학습에 넣을 것과 뺄 것을 먼저 가른다
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
import joblib
import numpy as np

DROP = ['OK', 'LOT', 'T']        # 정답과, 새 로트에는 없을 번호·시각
X = out.drop(columns=DROP).fillna(out.median(numeric_only=True))
y = out['OK']
print('열', list(X.columns))

`LOT` 과 `T` 를 뺀 이유가 있다. **다음 달 로트에는 그 번호가 없다.**
지금 데이터에서만 잘 맞고 실제로는 못 쓰는 열이다. 정답 `OK` 도 당연히 뺀다.

In [ ]:
# 가르고, 성적 찍는 함수를 만들고, 기준선부터 본다
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

def score(name, m):
    p = m.predict(Xte)
    print('%-14s 정확도 %.3f  불량 재현율 %.3f  불량 정밀도 %.3f'
          % (name, accuracy_score(yte, p), recall_score(yte, p, pos_label=0),
             precision_score(yte, p, pos_label=0, zero_division=0)))
    return p

print('학습 %d행 · 검증 %d행 · 검증 불량 %d건' % (len(Xtr), len(Xte), (yte == 0).sum()))
print('%-14s 정확도 %.3f  불량 재현율 0.000  ← 기준선' % ('전부 양품', (yte == 1).mean()))

In [ ]:
# 단순한 것부터 — 로지스틱과 랜덤포레스트
lr = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xtr, ytr)
_ = score('로지스틱', lr)
p_rf = score('랜덤포레스트', rf)

**로지스틱의 불량 재현율이 0.000 이다.** 정확도는 0.870 으로 기준선과 똑같다.
전부 양품이라고 찍은 것과 **한 글자도 다르지 않다**. 정확도만 봤으면 못 알아챈다.
랜덤포레스트는 0.387 — 있는 불량 62건 중 24건을 잡았다. 이쪽이 실제로 일을 한 것이다.

In [ ]:
# 무엇을 놓쳤는지 센다
cm = confusion_matrix(yte, p_rf, labels=[0, 1])
print('잡은 불량 %d건 · 놓친 불량 %d건' % (cm[0][0], cm[0][1]))
print('헛경보   %d건 · 맞은 양품 %d건' % (cm[1][0], cm[1][1]))

### 파일 하나로 남긴다

학습은 **한 번**, 판정은 **매번**이다. 그래서 갈라 둔다.
학습한 모델을 파일로 저장해 두면 다음부터는 불러 쓰기만 하면 된다.

In [ ]:
# 한 번 저장하고, 이후로는 불러 쓰기만 한다
joblib.dump({'model': rf, 'columns': list(X.columns)}, 'model.pkl')

bundle = joblib.load('model.pkl')
m, cols = bundle['model'], bundle['columns']
row = Xte.iloc[[0]][cols]                      # 실제로는 새로 들어온 한 줄
prob = m.predict_proba(row)[0][0]
print('불량 확률 %.3f → %s' % (prob, '불량 위험' if prob >= 0.5 else '양품'))

`columns` 를 같이 저장한 이유는 **열 순서가 어긋나면 조용히 틀리기** 때문이다.
판정할 때 같은 순서로 맞춰 넣는다.

**여기까지가 오늘의 전부다.** 표를 밖으로 한 줄도 안 보내고 `model.pkl` 하나를 만들었다.
웹 앱에 붙이는 것은 이 파일을 불러 쓰는 일뿐이다.

### 사내에 붙일 때 챙길 것

| 챙길 것 | 왜 |
|---|---|
| 데이터는 도구 안에 둔다 | 표를 프롬프트에 통째로 넣으면 그대로 반출이다 |
| 도구 하나는 일 하나만 | 여러 일을 하면 모델이 오용한다 |
| 에러는 고칠 방법까지 | 모델이 그 문구를 읽고 다시 고른다 |
| 되돌릴 수 없는 일은 확인 | 삭제 · 발주 · 발송은 사람에게 묻는다 |
| 부른 도구와 인자를 로그로 | 안 보이면 못 고친다 |

> **실습문제 4.** 판정 문턱을 **0.5 에서 0.3 으로** 낮춰 놓친 불량이 몇 건으로 주는지 본다.
> 숫자 하나만 바꾼다. 놓친 불량은 줄고 헛경보는 는다.

In [ ]:
# predict_proba 는 불량일 확률을 돌려준다
THRESHOLD = 0.3

# 문턱 0.5 에서는 잡은 24 · 놓친 38 · 헛경보 6 이었다
prob = rf.predict_proba(Xte)[:, 0]
pred = np.where(prob >= THRESHOLD, 0, 1)
cm2 = confusion_matrix(yte, pred, labels=[0, 1])
print('문턱 %.1f → 잡은 불량 %d · 놓친 불량 %d · 헛경보 %d'
      % (THRESHOLD, cm2[0][0], cm2[0][1], cm2[1][0]))